# conv-output-shape composite — cx3: parametric Conv2d output shape feeds einsum-over-flat-patches reduction

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `conv-output-shape`, `einops-einsum`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "conv-output-shape"
DD_ATOM_IDS = ["conv-output-shape", "einops-einsum"]
DD_SUBTOPICS = ["CNN: Conv output shape", "Einops: Deep Learning"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Two roles in a conv forward:
1. **conv-output-shape** — the analytic `(H_out, W_out)` formula. Pure integer math.
2. **einops-einsum** — the actual contraction that produces the conv output values.

The bridge: einsum's OUTPUT shape is derived from the labels on the right side of the string (`-> b o h w`), but the actual `H` and `W` dims of those output axes are determined by the input patch tensor's shape — which the caller built using `H_out` and `W_out` from the conv-output-shape formula.

This drill exercises BOTH atoms by:
- (a) asking you to compute `(H_out, W_out)` for a parametric `(stride, pad)` setting, then
- (b) running the einsum reduction over a PRE-COMPUTED `(B, C_in, H_out, W_out, K, K)`   patch tensor that the test passes you. Your job is JUST the einsum step — but you must   return `(H_out, W_out)` alongside the output for the test to verify.

This isolates the conv-output-shape arithmetic and the einsum contraction without dragging in `as_strided` mechanics, so the test can fuzz a wide range of `(stride, pad)` settings.

### Composite Exercise — parametric Conv2d output shape feeds einsum-over-flat-patches reduction

**Atoms exercised together**: `conv-output-shape`, `einops-einsum`

Implement `cx3_einsum_with_shape(patches, w, H, W, K, stride, pad)`.

- `patches`: float tensor of shape `(B, C_in, H_out, W_out, K, K)` — the test gives you the pre-built strided view; you do NOT need to construct it.
- `w`: float tensor of shape `(C_out, C_in, K, K)`.
- `H, W, K, stride, pad`: input dims and conv params (ints).

Return a tuple `((H_out, W_out), out)`:
- `(H_out, W_out)`: int output spatial dims from the conv-output-shape formula. The test asserts these match the analytic formula AND match `patches.shape[2:4]`.
- `out`: float tensor of shape `(B, C_out, H_out, W_out)` from one einsum call against `patches` and `w`.

1. **conv-output-shape atom** — `H_out = (H + 2*pad - K) // stride + 1` (same for W).
2. **einops-einsum atom** — `einops.einsum(patches, w, 'b c h w kh kw, o c kh kw -> b o h w')`.

In [ ]:
def cx3_einsum_with_shape(patches, w, H, W, K, stride, pad):
    # Atom A (conv-output-shape): closed-form analytic shape.
    H_out = (H + 2 * pad - K) // stride + 1
    W_out = (W + 2 * pad - K) // stride + 1
    # Atom B (einops-einsum): contract c, kh, kw between patches and weight.
    out = einops.einsum(patches, w, 'b c h w kh kw, o c kh kw -> b o h w')
    return (H_out, W_out), out


<details><summary>Show solution — cx3</summary>

```python
def cx3_einsum_with_shape(patches, w, H, W, K, stride, pad):
    # Atom A (conv-output-shape): closed-form analytic shape.
    H_out = (H + 2 * pad - K) // stride + 1
    W_out = (W + 2 * pad - K) // stride + 1
    # Atom B (einops-einsum): contract c, kh, kw between patches and weight.
    out = einops.einsum(patches, w, 'b c h w kh kw, o c kh kw -> b o h w')
    return (H_out, W_out), out
```

Returning `(H_out, W_out)` alongside `out` is the test's hook to verify both atoms independently: the analytic value AND its agreement with `patches.shape[2:4]`. In real ARENA code these two numbers ALWAYS line up — if they don't, you've miscomputed the conv shape and your einsum is silently producing garbage.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx3',
        'subtopics': ["CNN: Conv output shape", "Einops: Deep Learning"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()